In [2]:
import asyncio
import json
import os
import time
from pathlib import Path

import pandas as pd
import ollama
from openai import AsyncOpenAI

# Make sure your OPENAI_API_KEY is set in the environment
from dotenv import load_dotenv
import os

load_dotenv(r"C:\Users\prahn\OneDrive\Documents\IITM-Pravartak\Pravartak_Practice\practice_scripts\.env")
assert os.environ.get('OPENAI_API_KEY'), 'Set OPENAI_API_KEY first'

client = AsyncOpenAI()

MODEL = 'gpt-4o-mini'
JUDGE_MODEL = 'gpt-4o'
TEMPERATURE = 0.0

# Cost rates ($ per token) — from W4 cost.py
RATES = {
    'gpt-4o-mini': {'in': 0.15 / 1_000_000, 'out': 0.60 / 1_000_000},
    'gpt-4o':      {'in': 2.50 / 1_000_000, 'out': 10.00 / 1_000_000},
}

print('Setup complete.')

Setup complete.


## Step 1 — Load the data

In [3]:
DATA_DIR = Path('../data')   # adjust if your folder layout differs

snippets = [json.loads(line) for line in (DATA_DIR / 'job_snippets.jsonl').read_text().splitlines() if line.strip()]
golden = {row['id']: row for row in (json.loads(line) for line in (DATA_DIR / 'golden_set.jsonl').read_text().splitlines() if line.strip())}

print(f'Loaded {len(snippets)} snippets, {len(golden)} golden entries.')
print('Sample snippet:', snippets[0])

Loaded 10 snippets, 10 golden entries.
Sample snippet: {'id': 'j01', 'snippet': 'Acme Corp is hiring a Senior Software Engineer to join our platform team. The ideal candidate has 5+ years of backend development experience and strong skills in Python and distributed systems.'}


In [13]:
snippets[0]

{'id': 'j01',
 'snippet': 'Acme Corp is hiring a Senior Software Engineer to join our platform team. The ideal candidate has 5+ years of backend development experience and strong skills in Python and distributed systems.'}

## Step 2 — Write the four prompt strategies

Each strategy is a function that takes a snippet text and returns the messages list to send to the LLM.

Implement all four. Keep each one focused — the point is to *see* the difference between strategies, not to over-engineer any one.

**TODO:** fill in the four `prompt_*` functions below.

In [4]:
#Note: This does not return the id of the job. It must be extracted from the snippets list
def prompt_zero_shot(snippet_text: str) -> list[dict]:
    """Strategy 1 — zero-shot. Just ask, no examples, no persona."""

    zero_shot_prompt = f"""Based only on the job description snippet provided, extract and return the following information as JSON.
1. company — the company doing the hiring
2. role — the job title
3. years_experience_required — the minimum experience required (integer). If experience required is not specified, return null.

Snippet:
{snippet_text}
"""
    return [{'role': 'user', 'content': zero_shot_prompt}]
    

In [5]:
#Note: This does not return the id of the job. It must be extracted from the snippets list

def prompt_few_shot(snippet_text: str) -> list[dict]:
    """Strategy 2 — few-shot. Include 2-3 worked examples in the prompt."""
    few_shot_prompt = f"""
Using the three example job descriptions and their outputs as reference,
extract the following information from the job description provided:

- company
- role
- years_experience_required

Return the result as JSON. When experience is specified, years_experience_required
must be an integer. If experience required is not specified, return null

Example 1:
Job description:
ABC is recruiting a Senior Engineer to join their team. The ideal candidate
must have 10+ years of experience.

Output:
{{"company": "ABC", "role": "Senior Engineer", "years_experience_required": 10}}

Example 2:
Job description:
DEF Inc is recruiting an HR Manager to join their management. Applicants
must have three to five years of experience.

Output:
{{"company": "DEF Inc", "role": "HR Manager", "years_experience_required": 3}}

Example 3:
Job description:
GHI is recruiting an Analyst to join their data team. Freshers can also
apply if they can demonstrate experience.

Output:
{{"company": "GHI", "role": "Analyst", "years_experience_required": null}}

Now extract the information from this job description:

{snippet_text}
"""
    return [{'role': 'user', 'content': few_shot_prompt}]

In [6]:
def prompt_structured(snippet_text: str) -> list[dict]:
    """Strategy 3 — structured / role-based. Use a system prompt with a persona and explicit JSON schema."""
    structured_prompt = """
ROLE: You are an expert recruiter.
CONTEXT: The user needs specific details extracted from a job description provided as natural language text.
TASK: 
1. Analyze the provided job description
2. Extract the three fields:
 - "company": the company doing the hiring
 - "role": the job title
 - "years_experience_required": the minimum experience required (integer)
3. If experience required is not specified, return null
FORMAT: Return ONLY a valid JSON object using exactly this schema:
{
  "company": string,
  "role": string,
  "years_experience_required": integer or null
}
"""
    return [{'role': 'system', 'content': structured_prompt}, {'role': 'user', 'content': snippet_text}]    



In [7]:
def prompt_cot(snippet_text: str) -> list[dict]:
    """Strategy 4 — chain-of-thought. Ask the model to reason before answering."""
    cot_prompt = f"""Based only on the job description snippet provided, extract:
1. company — the company doing the hiring
2. role — the job title
3. years_experience_required — the minimum experience required (integer). 
Think step by step about the information in the job description before providing your final JSON answer.
When experience is specified, years_experience_required must be an integer. If experience required is not specified, return null.
Return the final answer as JSON.

Snippet:
{snippet_text}
"""
    return [{'role': 'user', 'content': cot_prompt}]

In [8]:
print(prompt_zero_shot(snippets[0]['snippet']))
print(prompt_few_shot(snippets[0]['snippet']))
print(prompt_structured(snippets[0]['snippet']))
print(prompt_cot(snippets[0]['snippet']))

[{'role': 'user', 'content': 'Based only on the job description snippet provided, extract and return the following information as JSON.\n1. company — the company doing the hiring\n2. role — the job title\n3. years_experience_required — the minimum experience required (integer). If experience required is not specified, return null.\n\nSnippet:\nAcme Corp is hiring a Senior Software Engineer to join our platform team. The ideal candidate has 5+ years of backend development experience and strong skills in Python and distributed systems.\n'}]
[{'role': 'user', 'content': '\nUsing the three example job descriptions and their outputs as reference,\nextract the following information from the job description provided:\n\n- company\n- role\n- years_experience_required\n\nReturn the result as JSON. When experience is specified, years_experience_required\nmust be an integer. If experience required is not specified, return null\n\nExample 1:\nJob description:\nABC is recruiting a Senior Engineer t

In [9]:
STRATEGIES = {
    'zero_shot': prompt_zero_shot,
    'few_shot': prompt_few_shot,
    'structured': prompt_structured,
    'cot': prompt_cot,
}

## ** STEP 2**

In [18]:
RATES = {
    'gpt-4o-mini': {'in': 0.15 / 1_000_000, 'out': 0.60 / 1_000_000},
    'gpt-4o':      {'in': 2.50 / 1_000_000, 'out': 10.00 / 1_000_000},
}

In [12]:
def parse_response(text: str) -> dict | None:
    """Try to parse a JSON object out of the model's response. Return None if it doesn't parse.
    
    Hint: models sometimes wrap JSON in ```json ... ``` fences. Strip them first.
    """

    text = text.strip()

    # Try plain JSON first
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        pass

    # Handle prose followed by ```json ... ```
    if "```json" in text:
        text = text.split("```json", 1)[1]
        text = text.split("```", 1)[0].strip()

        try:
            return json.loads(text)
        except json.JSONDecodeError:
            return None

    return None


In [22]:
is_local=False

def compute_cost_usd(RATES:dict, prompt_tokens: int, completion_tokens: int, model="gpt-4o-mini") -> float:
   
    rates = RATES.get(model)
    if rates is None:
        return 0.0
    model_rates = rates.get(model)

    if model_rates is None:
        return 0.0

    return round(prompt_tokens * model_rates["in"] + completion_tokens * model_rates["out"],  4 )


async def run_one(strategy_name: str, snippet: dict) -> dict:
    """Run one strategy on one snippet. Return a dict with all the captured fields."""
    snippet_ID = snippet['id']
    messages = STRATEGIES[strategy_name](snippet['snippet'])
    output = {}
    if is_local:
        ollama_client = ollama.AsyncClient()
        started = time.time()
        response = await ollama_client.chat( model='gemma3:4b',
        messages=messages,
        options={
        'temperature': 0.0},  
        )
        elapsed = round(time.time() - started, 3)
        raw_response = response['message']['content']
        output['strategy_name'] = strategy_name
        output['snippet_ID'] = snippet_ID
        output['raw_response'] = raw_response
        output['parsed_extraction'] = parse_response(raw_response)
        output['cost_USD'] = compute_cost_usd(RATES, prompt_tokens=response.prompt_eval_count, completion_tokens=response.eval_count)
        output['latency_s'] = elapsed
        
        return output
    else:
       
        started = time.time()
    
        response = await client.chat.completions.create(
            model="gpt-4o-mini",
            messages=messages,
            temperature=0.0,
        )

        elapsed = round(time.time() - started, 3)

        raw_response = response.choices[0].message.content

        output['strategy'] = strategy_name
        output['snippet_id'] = snippet_ID
        output['raw_response'] = raw_response
        output['parsed_extraction'] = parse_response(raw_response)

        output['cost_usd'] = compute_cost_usd(
            RATES,
            prompt_tokens=response.usage.prompt_tokens,
            completion_tokens=response.usage.completion_tokens
        )

        output['latency_s'] = elapsed

        return output
   

In [23]:
op = await run_one('zero_shot', snippets[0])
print(op)

AttributeError: 'tuple' object has no attribute 'get'

In [39]:
type(op.get('parsed_response'))

dict

In [15]:

async def run_all(strategies, snippets) -> list[dict]:
    """Run all 10 × 4 = 40 calls in parallel. Use asyncio.gather."""
    tasks = []
    for key in strategies.keys():
        for i in range (len(snippets)):
            tasks.append(run_one(key, snippets[i]))
    results = await asyncio.gather(*tasks)
    return results

In [16]:
results = await run_all(STRATEGIES, snippets)

AttributeError: 'tuple' object has no attribute 'get'

In [1]:
print(results[38])

NameError: name 'results' is not defined

In [40]:
import pandas as pd
results_df = pd.DataFrame(results)
results_df
results_df.to_csv("results.csv", index=False)

In [41]:
results_df

,strategy,snippet_id,raw_response,parsed_extraction,cost_usd,latency_s
0,zero_shot,j01,"```json\n{\n ""company"": ""Acme Corp"",\n ""role...","{'company': 'Acme Corp', 'role': 'Senior Softw...",0.0000,2.534
1,zero_shot,j02,"```json\n{\n ""company"": ""Northwind Ltd."",\n ...","{'company': 'Northwind Ltd.', 'role': 'Data An...",0.0000,1.813
2,zero_shot,j03,"```json\n{\n ""company"": ""Globex International...","{'company': 'Globex International', 'role': 'P...",0.0000,2.192
3,zero_shot,j04,"```json\n{\n ""company"": ""Initech"",\n ""role"":...","{'company': 'Initech', 'role': 'Lead DevOps En...",0.0000,1.916
4,zero_shot,j05,"```json\n{\n ""company"": ""Hooli"",\n ""role"": ""...","{'company': 'Hooli', 'role': 'Junior Frontend ...",0.0000,2.357
5,zero_shot,j06,"```json\n{\n ""company"": ""Pied Piper Inc."",\n ...","{'company': 'Pied Piper Inc.', 'role': 'Senior...",0.0000,2.579
6,zero_shot,j07,"```json\n{\n ""company"": ""Soylent Industries"",...","{'company': 'Soylent Industries', 'role': 'Eng...",0.0000,2.101
7,zero_shot,j08,"```json\n{\n ""company"": ""Wonka Confectionery ...","{'company': 'Wonka Confectionery Ltd.', 'role'...",0.0000,1.882
8,zero_shot,j09,"```json\n{\n ""company"": ""Stark Industries"",\n...","{'company': 'Stark Industries', 'role': 'Cyber...",0.0000,1.877
9,zero_shot,j10,"```json\n{\n ""company"": ""Cyberdyne Systems"",\...","{'company': 'Cyberdyne Systems', 'role': 'AI/M...",0.0000,2.100


In [ ]:
for i in range(len(results_df)):
    print(f"Row {i}:")
    print("Parsed extraction:",type(results_df['parsed_extraction'][0]))
    


## ** This is the modified parser response**

In [56]:
import json

def parse_response(text: str) -> dict | None:
    """Try to parse a JSON object out of the model's response."""

    text = text.strip()

    # Try plain JSON first
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        pass

    # Handle prose followed by ```json ... ```
    if "```json" in text:
        text = text.split("```json", 1)[1]
        text = text.split("```", 1)[0].strip()

        try:
            return json.loads(text)
        except json.JSONDecodeError:
            return None

    return None

In [59]:
for i in range(len(results)):
    parsed = parse_response(results[i]["raw_response"])
    #print(parsed)
    #print(type(parsed))
    print("Parsed extraction:", parsed)

Parsed extraction: {'company': 'Acme Corp', 'role': 'Senior Software Engineer', 'years_experience_required': 5}
Parsed extraction: {'company': 'Northwind Ltd.', 'role': 'Data Analyst', 'years_experience_required': 2}
Parsed extraction: {'company': 'Globex International', 'role': 'Product Manager - Growth', 'years_experience_required': 4}
Parsed extraction: {'company': 'Initech', 'role': 'Lead DevOps Engineer', 'years_experience_required': 6}
Parsed extraction: {'company': 'Hooli', 'role': 'Junior Frontend Developer', 'years_experience_required': None}
Parsed extraction: {'company': 'Pied Piper Inc.', 'role': 'Senior ML Engineer', 'years_experience_required': 7}
Parsed extraction: {'company': 'Soylent Industries', 'role': 'Engineering Manager', 'years_experience_required': 3}
Parsed extraction: {'company': 'Wonka Confectionery Ltd.', 'role': 'Senior UX Researcher', 'years_experience_required': 5}
Parsed extraction: {'company': 'Stark Industries', 'role': 'Cybersecurity Analyst', 'years_

In [53]:
plain = "{  ""company"": ""Wonka Confectionery Ltd."",  ""role"": ""Senior UX Researcher"",  ""years_experience_required"": 5}"
parsed = parse_response(plain)
type(parsed)  # Should be <class 'dict'>
#print(parsed)

NoneType